In [ ]:
# ==============================================================================
# TRỢ LÝ AI PHÁP LUẬT GIAO THÔNG
# ==============================================================================
# 0. ĐỒNG BỘ HÓA THƯ VIỆN & VÁ LỖI HỆ THỐNG
!pip install -q -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -q -U chromadb langchain-chroma transformers accelerate bitsandbytes langchain-huggingface gradio

import os
import json
import torch
import gradio as gr

# 1. KẾT NỐI GOOGLE DRIVE
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("[*] Đang kết nối Google Drive...")
        drive.mount('/content/drive')
except ImportError:
    print("[*] Không chạy trên Colab, bỏ qua bước mount Drive.")

# Import các thư viện lõi
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_chroma import Chroma
from langchain_core.stores import InMemoryStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# 2. CẤU HÌNH ĐƯỜNG DẪN
BASE_PATH = "/content/drive/MyDrive/NLP"
DATASET_DIR = os.path.join(BASE_PATH, "data/datasets")
VECTOR_DB_DIR = os.path.join(BASE_PATH, "vector_db_parent_child")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
PARENT_JSON = os.path.join(DATASET_DIR, "parents_traffic_law.json")

def main():
    # 3. KHỞI TẠO HỆ THỐNG TRUY XUẤT (KNOWLEDGE BASE)
    print("\n[*] Đang nạp Knowledge Base từ Drive...")
    if not os.path.exists(PARENT_JSON):
        print(f"[!] LỖI: Không tìm thấy {PARENT_JSON}. Hãy kiểm tra lại file trên Drive.")
        return

    with open(PARENT_JSON, "r", encoding="utf-8") as f:
        parents_data = json.load(f)

    docstore = InMemoryStore()
    docstore.mset([(p["doc_id"], Document(page_content=p["content"], metadata=p.get("metadata", {}))) for p in parents_data])

    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    vectorstore = Chroma(
        collection_name="child_chunks_collection",
        embedding_function=embedding_model,
        persist_directory=VECTOR_DB_DIR
    )

    def retrieve_context(query):
        # Tìm 3 đoạn luật có liên quan nhất
        child_docs = vectorstore.similarity_search(query, k=3)
        parent_ids = list(set([d.metadata.get("parent_id") for d in child_docs if "parent_id" in d.metadata]))
        docs = docstore.mget(parent_ids)
        context = "\n\n".join([f"--- [Nguồn: {d.metadata.get('article', 'Luật')}] ---\n{d.page_content}" for d in docs if d])
        return context[:3500] # Tối ưu độ dài Context để mô hình đọc tốt nhất

    # 4. TẢI MÔ HÌNH
    print("[*] Đang nạp Qwen-3B-Instruct vào GPU...")
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    stop_token_ids = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto"
    )

    model_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512, # Tăng nhẹ độ dài để trả lời đầy đủ
        temperature=0.1,    # Giữ độ chính xác cao
        top_p=0.9,
        do_sample=True,
        eos_token_id=stop_token_ids,
        return_full_text=False
    )
    hf_llm = HuggingFacePipeline(pipeline=model_pipeline)

    # 5. RAG CHAIN
    # Loại bỏ các tiền tố dư thừa như "Theo tài liệu..." để câu trả lời chuyên nghiệp hơn
    strict_qwen_prompt = """<|im_start|>system
Bạn là chuyên gia tư vấn Pháp luật Giao thông Việt Nam.
Nhiệm vụ của bạn là giải đáp thắc mắc của người dân một cách chính xác, ngắn gọn và lịch sự, chỉ sử dụng Tiếng việt cho câu trả lời.
QUY TẮC:
1. CHỈ sử dụng thông tin trong [TÀI LIỆU CĂN CỨ] để trả lời.
2. Không sử dụng các cụm từ như "Theo tài liệu hiện có" hay "Dựa trên văn bản" mà thay vào đó là "Theo điều luật số...".
3. Nếu không có thông tin trong tài liệu, hãy đáp: "Dựa trên hệ thống luật hiện hành, chúng tôi chưa có quy định cụ thể về trường hợp này."
4. CẤM tự bịa đặt thông tin hoặc mức phạt.<|im_end|>
<|im_start|>user
[TÀI LIỆU CĂN CỨ]:
{context}

[CÂU HỎI]:
{question}<|im_end|>
<|im_start|>assistant
"""
    rag_chain = (
        {"context": RunnableLambda(retrieve_context), "question": RunnablePassthrough()}
        | PromptTemplate.from_template(strict_qwen_prompt)
        | hf_llm
        | StrOutputParser()
    )

    # 6. GIAO DIỆN CHATBOT CHUYÊN NGHIỆP
    def chatbot_response(message, history):
        response = rag_chain.invoke(message)
        return response.strip()

    print("[*] Đang khởi chạy giao diện Demo...")
    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("<h1 style='text-align: center; color: #1A5276;'>HỆ THỐNG TƯ VẤN PHÁP LUẬT GIAO THÔNG</h1>")
        gr.Markdown("<p style='text-align: center;'>Hỗ trợ tra cứu Luật Giao thông Đường bộ và Nghị định 100 bằng trí tuệ nhân tạo (RAG-Engine).</p>")

        # Tùy chỉnh Chatbot: Ẩn hoàn toàn chữ "Chatbot" ở góc trái trên cùng
        custom_chatbot = gr.Chatbot(show_label=False)

        # Sử dụng ChatInterface với các câu hỏi được đặt ngay bên dưới khung chat
        gr.ChatInterface(
            fn=chatbot_response,
            chatbot=custom_chatbot,
            examples=[
                "Đường bộ bao gồm những thành phần nào?",
                "Trên đường có nhiều làn đường, khi chuyển làn đường cần lưu ý điều kiện gì?",
                "Bao nhiêu tuổi thì được cấp Giấy phép lái xe hạng B1, B2?",
                "Đường phố gồm những phần nào?"
            ],
        )

    demo.launch(share=True, inline=True)

if __name__ == "__main__":
    main()


[*] Đang nạp Knowledge Base từ Drive...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[*] Đang nạp Qwen-3B-Instruct vào GPU...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[*] Đang khởi chạy giao diện Demo...


/tmp/ipykernel_4117/649970348.py:125: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1533a6d68715a0fa81.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
